# Le déterminant comme facteur d'aire

| Module | Durée | Prérequis |
|---|---|---|
| M4 – Inverse, déterminant, rang | 75 min | M2 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

L'association des jardins partagés du quartier a reçu un plan de ses parcelles. Pour qu'il soit plus joli, le dessinateur l'a tracé en **perspective cavalière** : le terrain y apparaît incliné et écrasé. La légende du plan indique :

- 1 m vers l'**est** sur le terrain correspond à 1 cm vers la droite sur le plan ;
- 1 m vers le **nord** sur le terrain correspond à 0,5 cm vers la droite et 0,6 cm vers le haut sur le plan.

La trésorière a mesuré sur le plan la surface de chaque parcelle, en cm². Elle doit facturer l'eau au mètre carré **réel**.

**La question de la trésorière : comment passer de la surface mesurée sur le plan à la surface réelle ? Et ce rapport est-il le même pour toutes les parcelles, quelle que soit leur forme ?**

**Activité** : Quelles informations de la légende sont nécessaires ? Le plan conserve-t-il les longueurs ? les angles ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

La parcelle des Tomates mesure 18 cm² sur le plan.

In [ ]:
pari_surface_tomates = 30     # surface réelle en m² (modifiez)

## ③ Abstraire : le déterminant

D'après le module M3, la légende décrit une transformation linéaire : ses colonnes sont les images des directions est et nord.

$$
A = \begin{pmatrix} 1 & 0{,}5 \\ 0 & 0{,}6 \end{pmatrix} \quad \text{(en cm par mètre)}
$$

Le carré unité du terrain (1 m²) devient sur le plan le parallélogramme construit sur les deux colonnes. Son aire est le **déterminant** de la matrice :

$$
\det \begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc
$$

![Schéma : le parallélogramme construit sur les colonnes a pour aire ad - bc ; un déterminant négatif retourne la figure, un déterminant nul l'aplatit](../../assets/schemas/schema-determinant-aire.svg)

Comme la transformation est linéaire, **toutes** les surfaces sont multipliées par le même facteur, quelle que soit leur forme :

$$
\text{aire sur le plan} = \lvert \det A \rvert \times \text{aire réelle}
$$

Le signe du déterminant indique si la figure est retournée. Trois propriétés suivent de cette lecture géométrique :

$$
\det(AB) = \det A \times \det B, \qquad \det(A^{-1}) = \frac{1}{\det A}, \qquad \det A = 0 \iff A \text{ n'est pas inversible.}
$$

Pour une matrice $2 \times 2$ de déterminant non nul, l'inverse s'écrit :

$$
A^{-1} = \frac{1}{ad - bc} \begin{pmatrix} d & -b \\ -c & a \end{pmatrix}
$$

**Activité** : Calculez $\det A$ pour le plan, puis la surface réelle des Tomates. Calculez aussi $A^{-1}$ et vérifiez que $A A^{-1} = I$.

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Le déterminant, à la main et avec NumPy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def det2(A):
    (a, b), (c, d) = A
    return a * d - b * c

A = np.array([[1, 0.5], [0, 0.6]])
print("det2(A)          :", det2(A))
print("np.linalg.det(A) :", np.linalg.det(A))
print("A^-1 :\n", np.linalg.inv(A))
print("A A^-1 = I ?", np.allclose(A @ np.linalg.inv(A), np.eye(2)))

### Les parcelles

La trésorière a relevé sur le plan les sommets de chaque parcelle (en cm). On calcule leur aire par la formule du lacet, vue au notebook 3.1.

In [ ]:
def aire(polygone):
    # Formule du lacet ; le polygone est fermé (dernier point = premier point)
    x, y = polygone
    return 0.5 * abs(np.dot(x[:-1], y[1:]) - np.dot(x[1:], y[:-1]))

def ferme(points):
    points = np.asarray(points, dtype=float).T
    return np.hstack([points, points[:, :1]])

parcelles_plan = {
    "Tomates":     ferme([(0, 0), (6, 0), (8.5, 3), (2.5, 3)]),
    "Courges":     ferme([(7, 0), (12, 0), (15, 3.6), (10, 3.6)]),
    "Aromatiques": ferme([(3.5, 4.2), (9, 4.2), (6.5, 6.6)]),
    "Fraises":     ferme([(10.5, 4.2), (16, 4.2), (17.6, 6.12), (14.6, 7.32), (12.1, 7.32)]),
}
for nom, p in parcelles_plan.items():
    print(f"{nom:12s} : {aire(p):6.2f} cm² sur le plan")

### Redresser le plan

Pour retrouver le terrain réel, on applique la transformation **inverse** : un point du plan $\mathbf{p}$ provient du point réel $A^{-1}\mathbf{p}$.

In [ ]:
A_inv = np.linalg.inv(A)
parcelles_reelles = {nom: A_inv @ p for nom, p in parcelles_plan.items()}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, parcelles, titre, unite in [(axes[0], parcelles_plan, "Le plan (perspective cavalière)", "cm"),
                                    (axes[1], parcelles_reelles, "Le terrain réel, redressé par A⁻¹", "m")]:
    for k, (nom, p) in enumerate(parcelles.items()):
        couleur = [LIOUAI["vert"], LIOUAI["gris"], LIOUAI["jaune"], LIOUAI["vert_texte"]][k]
        ax.fill(*p, color=couleur, alpha=0.6); ax.plot(*p, color=LIOUAI["navy"], lw=1)
        ax.text(*p[:, :-1].mean(axis=1), nom, ha="center", va="center", fontsize=9)
    ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.set_title(titre); ax.set_xlabel(unite); ax.set_ylabel(unite)
plt.tight_layout(); plt.show()

print(f"{'parcelle':12s} {'plan (cm²)':>11s} {'réel (m²)':>10s} {'rapport':>8s}")
for nom in parcelles_plan:
    s_plan, s_reel = aire(parcelles_plan[nom]), aire(parcelles_reelles[nom])
    print(f"{nom:12s} {s_plan:11.2f} {s_reel:10.2f} {s_plan / s_reel:8.3f}")

### Les propriétés, vérifiées sur des exemples

In [ ]:
rng = np.random.default_rng(0)
for _ in range(3):
    X, Y = rng.normal(size=(2, 2)), rng.normal(size=(2, 2))
    print(f"det(XY) = {np.linalg.det(X @ Y):8.4f}    det(X) det(Y) = {np.linalg.det(X) * np.linalg.det(Y):8.4f}")

print("\nLe catalogue du notebook 3.1 :")
t = np.radians(30)
catalogue = {"Pivoter de 30°": [[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]], "Agrandir × 1,5": [[1.5, 0], [0, 1.5]],
             "Étirer × 2": [[2, 0], [0, 1]], "Incliner": [[1, 0.4], [0, 1]], "Aplatir": [[1, 0], [0, 0]], "Miroir": [[-1, 0], [0, 1]]}
for nom, B in catalogue.items():
    print(f"  {nom:16s} det = {np.linalg.det(B):6.2f}")

### En dimension 3 : un facteur de volume

Pour une matrice $3 \times 3$, le déterminant est le volume (signé) du parallélépipède construit sur les trois colonnes. On peut le comparer au **produit mixte** $(\mathbf{u} \times \mathbf{v}) \cdot \mathbf{w}$, que les physiciens utilisent pour calculer un volume.

In [ ]:
u, v, w = np.array([2, 0, 0]), np.array([1, 3, 0]), np.array([0.5, 1, 4])
B3 = np.column_stack([u, v, w])
print("det        :", np.linalg.det(B3))
print("produit mixte :", np.dot(np.cross(u, v), w))

## ⑤ Interpréter

In [ ]:
print(f"Votre pari pour les Tomates : {pari_surface_tomates} m² ; surface réelle : {aire(parcelles_reelles['Tomates']):.1f} m²")

**Activité** :

1. Rédigez pour la trésorière une règle d'une phrase : comment convertir une surface du plan en surface réelle ?
2. Le rapport est-il le même pour toutes les parcelles ? Le triangle des Aromatiques fait-il exception ? Pourquoi ?
3. Dans le catalogue du notebook 3.1, quel est le déterminant du miroir ? Que signifie son signe sur le F ?
4. Le bouton « Aplatir » a un déterminant nul. Relisez la propriété $\det(A^{-1}) = 1 / \det A$ : pourquoi ne peut-on pas défaire un aplatissement ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse fragile.** On a supposé que le dessin est une transformation linéaire **exacte**. Un plan dessiné à la main ou photographié de travers ne l'est qu'approximativement. Si le coefficient 0,6 de la légende était en réalité 0,58, quelle erreur commettrait-on sur la facture d'eau ?
- **Vérifier autrement.** Retrouvez la surface réelle des Courges sans le déterminant : sur le terrain redressé, c'est un rectangle ; calculez ses côtés réels.
- **Généraliser.** Écrivez une fonction `surface_reelle(polygone_plan, A)` qui n'utilise **que** le déterminant, sans redresser le polygone. Ajoutez un message d'erreur si la légende est dégénérée.

In [ ]:
def surface_reelle(polygone_plan, A):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def surface_reelle(polygone_plan, A):
    d = np.linalg.det(A)
    if np.isclose(d, 0):
        raise ValueError("Légende dégénérée : le plan écrase le terrain sur une droite, on ne peut rien retrouver.")
    return aire(polygone_plan) / abs(d)

for nom, p in parcelles_plan.items():
    print(f"{nom:12s} : {surface_reelle(p, A):6.2f} m²")

A_bis = np.array([[1, 0.5], [0, 0.58]])
vraie = surface_reelle(parcelles_plan["Tomates"], A_bis)
calculee = surface_reelle(parcelles_plan["Tomates"], A)
print(f"\nSi le vrai coefficient est 0,58, les surfaces calculées avec 0,6 sont trop petites de {100 * (1 - calculee / vraie):.1f} %")

reel = parcelles_reelles["Courges"]
base = np.linalg.norm(reel[:, 1] - reel[:, 0]); hauteur = reel[1, 2] - reel[1, 1]
print(f"Courges : base {base:.2f} m × hauteur {hauteur:.2f} m = {base * hauteur:.2f} m²")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Décrire le concept": 1,
    "Relier les traits du résultat à leur sens réel": 1,
    "Estimer la solution avant de calculer": 1,
    "Expliquer visuellement": 1,
    "Avoir l'intuition du comportement de l'outil": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "4.1-determinant-et-aire", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")